# COMPASS multivariate models

Elastic-net Cox and XGBoost survival:cox, each in "both" (full feature set)
and "baseline" (androgen-axis-only) configurations, at every landmark.
Requires `01_preprocessing.ipynb` to have built the merged `profile_data`
inputs under `prediction_inputs_<arm>/` first.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc", "avpc")
COHORTS = (
    "all",
    "metastatic",          # retrospective ADT-intent strata
    "localized",
    "llm_metastatic",      # LLM stage-pipeline metastatic status
    "llm_nonmetastatic",
)
# Orthogonal to COHORTS: "none" keeps every patient, "pre_adt_castrate"
# drops those with a castrate testosterone (<50 ng/dL) before ADT start,
# who were presumably androgen-deprived elsewhere first.
EXCLUSIONS = ("none", "pre_adt_castrate")
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.FORCE_RERUN = OVERWRITE
RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS)

## Run multivariate models

Elastic-net (both/baseline) and XGBoost (both/baseline) arms. Set
`OVERWRITE = True` in the configuration cell to refit and replace existing outputs. With `False`, completed tasks are skipped.

In [ ]:
for run in RUNS:
    cp.run_multivariate(run)

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), then combined across runs.

In [ ]:
summary_dfs = {cp.run_key(run): cp.summarize_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_summary_df